# Experimentos 1, 2 e 4 — Llama 3.3 70B Instruct (V3)

**Modelo:** meta-llama/Llama-3.3-70B-Instruct  
**Experimentos:** Zero-Shot (Exp1), Few-Shot (Exp2), Chain of Thought (Exp4)  
**Folds:** 5 folds com media e desvio padrao  
**Split:** 80/20  

**Nota:** Exp3 (Fine-Tuning) e Exp5 (Instruction Tuning) nao incluidos neste notebook.  
LoRA em modelos de 70B+ excede o tempo maximo de sessao do Colab (estimativa: 6-7 dias por experimento).  
Documentado como limitacao tecnica de hardware.

**Checkpoints:** salvos a cada 50 redacoes por fold. Retomada automatica ao reexecutar.

In [ ]:
# Celula 1 - Instalacao
!pip install -q transformers accelerate bitsandbytes gdown scikit-learn matplotlib peft optuna
print('Instalado!')

In [ ]:
# Celula 2 - Imports e autenticacao
import re, gc, json, time, random, os
import numpy as np
import pandas as pd
import torch
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error,
    cohen_kappa_score, f1_score
)
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from google.colab import userdata
from huggingface_hub import login

optuna.logging.set_verbosity(optuna.logging.WARNING)

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

NOME_MODELO   = 'meta-llama/Llama-3.3-70B-Instruct'
NOME_CURTO    = 'llama70b'
N_FOLDS       = 5
RANDOM_STATE  = 42

print('Autenticado!')

In [ ]:
# Celula 3 - Dataset e K-Fold (80/20)
import gdown
import pandas as pd
import re
import ast
from sklearn.model_selection import StratifiedKFold

novo_id = '1chJZo8L4s3Zuv1nzHrZa3b6ycf0ePQhv'
gdown.download(
    f'https://drive.google.com/uc?id={novo_id}',
    'meu_dataset.csv', quiet=False
)
df_enem = pd.read_csv('meu_dataset.csv')

df_enem[['c1', 'c2', 'c3', 'c4', 'c5']] = df_enem['competence'].apply(ast.literal_eval).tolist()


def limpar(texto):
    if pd.isna(texto): return ''
    texto = str(texto).strip("[]'\" ")
    texto = texto.replace('\n', ' ')
    texto = re.sub(r'\[[A-Z/]+\]', '', texto)
    texto = re.sub(r'\{[a-z]+\}', '', texto)
    return re.sub(r'\s+', ' ', texto).strip()


df_enem['essay_limpo'] = df_enem['essay'].apply(limpar)
df_enem = df_enem[df_enem['score'] > 0].reset_index(drop=True)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
df_enem['fold'] = -1
for i, (tr, te) in enumerate(skf.split(df_enem, df_enem['score'])):
    df_enem.loc[te, 'fold'] = i

print(f'Total: {len(df_enem)} | Folds: {N_FOLDS} | Teste por fold: ~{len(df_enem) // N_FOLDS}')

In [ ]:
# Celula 4 - Funcoes auxiliares

def extrair_notas(resposta):
    try:
        texto = re.sub(r'```json|```', '', str(resposta))
        i = texto.find('{')
        j = texto.rfind('}') + 1
        if i == -1 or j <= i: return None
        dados = json.loads(texto[i:j])
        comps = ['C1', 'C2', 'C3', 'C4', 'C5']
        if all(c in dados for c in comps):
            if max(dados[c] for c in comps) <= 20:
                for c in comps: dados[c] *= 10
            dados['Nota_Total'] = sum(dados[c] for c in comps)
            return dados
        if 'Nota_Total' in dados: return dados
    except: pass
    return None


def extrair_notas_markdown(resposta):
    try:
        texto = str(resposta)
        comps = {}
        for c in ['C1', 'C2', 'C3', 'C4', 'C5']:
            m = re.search(rf'{c}[^:]*:\s*(\d+)', texto)
            if m: comps[c] = int(m.group(1))
        if len(comps) == 5:
            if max(comps.values()) <= 20:
                for c in comps: comps[c] *= 10
            comps['Nota_Total'] = sum(comps.values())
            return comps
        m = re.search(r'(?:Total|Nota\s*Total|Nota\s*Final)[^\d]*(\d{3,4})', texto, re.I)
        if m: return {'Nota_Total': int(m.group(1))}
    except: pass
    return None


def extrair(resposta):
    return extrair_notas(resposta) or extrair_notas_markdown(resposta)


def extrair_feedback(resposta):
    try:
        t = str(resposta)
        j = t.rfind('}')
        fb = t[j + 1:].strip() if j >= 0 else ''
        if len(fb) < 20:
            inicio = t.find('{')
            fb = t[:inicio].strip() if inicio >= 0 else ''
        return fb if len(fb) > 20 else 'Feedback nao disponivel'
    except:
        return 'Feedback nao disponivel'


def calcular_metricas(y_true, y_pred, nome):
    yt = np.array(y_true, dtype=float)
    yp = np.array(y_pred, dtype=float)
    mae  = mean_absolute_error(yt, yp)
    rmse = float(np.sqrt(mean_squared_error(yt, yp)))
    def disc(n): return np.clip(np.round(np.array(n) / 40).astype(int), 0, 25)
    try:    qwk = cohen_kappa_score(disc(yt), disc(yp), weights='quadratic')
    except: qwk = float('nan')
    try:    f1  = f1_score(disc(yt), disc(yp), average='weighted', zero_division=0)
    except: f1  = float('nan')
    print(f'\n{"="*55}')
    print(f'METRICAS — {nome}')
    print(f'{"="*55}')
    print(f'  Amostras : {len(yt)}')
    print(f'  MAE      : {mae:.4f}')
    print(f'  RMSE     : {rmse:.4f}')
    print(f'  QWK      : {qwk:.4f}')
    print(f'  F1 Score : {f1:.4f}')
    print(f'{"="*55}')
    return {'modelo': nome, 'mae': mae, 'rmse': rmse, 'qwk': qwk, 'f1': f1, 'n': len(yt)}


print('Funcoes auxiliares carregadas!')

In [ ]:
# Celula 5 - Prompts

def selecionar_exemplos_fs(df_treino):
    ex_baixo = df_treino[df_treino['score'].between(80, 300)].iloc[0]
    ex_medio = df_treino[df_treino['score'].between(400, 600)].iloc[0]
    ex_alto  = df_treino[df_treino['score'].between(700, 1000)].iloc[0]
    return [ex_baixo, ex_medio, ex_alto]


def gerar_prompt_zs(redacao, max_chars=None):
    if max_chars:
        redacao = redacao[:max_chars]
    return (
        'Voce e um avaliador oficial de redacoes do ENEM.\n'
        'Avalie a redacao seguindo a escala oficial: 0, 40, 80, 120, 160 ou 200 pontos por competencia.\n\n'
        'COMPETENCIAS:\n'
        '- C1 (Norma Culta): dominio da norma padrao (0-200)\n'
        '- C2 (Tema/Estrutura): adequacao ao tema e estrutura (0-200)\n'
        '- C3 (Argumentacao): selecao e organizacao de argumentos (0-200)\n'
        '- C4 (Coesao): uso de mecanismos linguisticos (0-200)\n'
        '- C5 (Proposta de Intervencao): proposta com agente, acao, meio, efeito (0-200)\n\n'
        'REDACAO:\n' + redacao + '\n\n'
        'Responda APENAS com o JSON:\n'
        '{"C1": valor, "C2": valor, "C3": valor, "C4": valor, "C5": valor, "Nota_Total": soma}'
    )


def gerar_prompt_fs(redacao, exemplos, max_chars=None, n_exemplos=3):
    if max_chars:
        redacao = redacao[:max_chars]
    bloco = ''
    for j, ex in enumerate(exemplos[:n_exemplos]):
        notas_ex = (
            '{"C1": ' + str(ex['c1']) +
            ', "C2": ' + str(ex['c2']) +
            ', "C3": ' + str(ex['c3']) +
            ', "C4": ' + str(ex['c4']) +
            ', "C5": ' + str(ex['c5']) +
            ', "Nota_Total": ' + str(ex['score']) + '}'
        )
        bloco += (
            '--- EXEMPLO ' + str(j + 1) +
            ' (score=' + str(ex['score']) + ') ---\n'
            'REDACAO: "' + ex['essay_limpo'][:300] + '..."\n'
            'AVALIACAO: ' + notas_ex + '\n\n'
        )
    return (
        'Voce e um avaliador oficial de redacoes do ENEM.\n'
        'Avalie usando a escala: 0, 40, 80, 120, 160 ou 200 por competencia.\n\n'
        'COMPETENCIAS:\n'
        '- C1 (Norma Culta): 0-200\n'
        '- C2 (Tema/Estrutura): 0-200\n'
        '- C3 (Argumentacao): 0-200\n'
        '- C4 (Coesao): 0-200\n'
        '- C5 (Proposta de Intervencao): 0-200\n\n'
        'EXEMPLOS AVALIADOS POR HUMANOS:\n' + bloco +
        'REDACAO A AVALIAR:\n"' + redacao + '"\n\n'
        'Responda APENAS com o JSON:\n'
        '{"C1": valor, "C2": valor, "C3": valor, "C4": valor, "C5": valor, "Nota_Total": soma}'
    )


def gerar_prompt_cot(redacao, max_chars=None):
    if max_chars:
        redacao = redacao[:max_chars]
    return (
        'Voce e um avaliador oficial de redacoes do ENEM.\n'
        'Avalie a redacao seguindo a escala: 0, 40, 80, 120, 160 ou 200 pontos por competencia.\n\n'
        'COMPETENCIAS:\n'
        '- C1 (Norma Culta): dominio da norma padrao (0-200)\n'
        '- C2 (Tema/Estrutura): adequacao ao tema e estrutura (0-200)\n'
        '- C3 (Argumentacao): selecao e organizacao de argumentos (0-200)\n'
        '- C4 (Coesao): uso de mecanismos linguisticos (0-200)\n'
        '- C5 (Proposta de Intervencao): proposta com agente, acao, meio, efeito (0-200)\n\n'
        'REDACAO:\n' + redacao + '\n\n'
        'Siga EXATAMENTE estas etapas:\n\n'
        'PASSO 1 - ANALISE:\nAnalise cada competencia separadamente.\n\n'
        'PASSO 2 - NOTAS:\nAtribua as notas no formato JSON:\n'
        '{"C1": valor, "C2": valor, "C3": valor, "C4": valor, "C5": valor, "Nota_Total": soma}\n\n'
        'PASSO 3 - FEEDBACK:\nEscreva 2-3 paragrafos de feedback construtivo.'
    )


print('Prompts carregados!')

In [ ]:
# Celula 6 - Inferencia local

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16
)


def carregar_modelo(nome_modelo):
    print(f'Carregando {nome_modelo}...')
    tok = AutoTokenizer.from_pretrained(nome_modelo, trust_remote_code=True)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    m = AutoModelForCausalLM.from_pretrained(
        nome_modelo, quantization_config=bnb_config,
        device_map='auto', trust_remote_code=True
    )
    m.eval()
    print('Carregado!')
    return tok, m


def liberar(m, tok):
    del m, tok
    gc.collect()
    torch.cuda.empty_cache()
    print('Memoria GPU liberada.')


def inf_local(tok, m, prompt, temp=0.1, max_tok=300):
    msgs = [{'role': 'user', 'content': prompt}]
    try:
        txt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    except:
        txt = prompt
    inp = tok(txt, return_tensors='pt', truncation=True, max_length=3072).to(m.device)
    with torch.no_grad():
        out = m.generate(
            **inp, max_new_tokens=max_tok,
            temperature=temp, do_sample=True,
            pad_token_id=tok.pad_token_id
        )
    return tok.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True).strip()


def inf_local_cot(tok, m, prompt, temp=0.2, max_tok=700):
    msgs = [{'role': 'user', 'content': prompt}]
    try:
        txt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    except:
        txt = prompt
    inp = tok(txt, return_tensors='pt', truncation=True, max_length=3072).to(m.device)
    with torch.no_grad():
        out = m.generate(
            **inp, max_new_tokens=max_tok,
            temperature=temp, do_sample=True,
            pad_token_id=tok.pad_token_id
        )
    return tok.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True).strip()


print('Funcoes de inferencia carregadas!')

In [ ]:
# Celula 7 - Runners com checkpoint por fold

def ckpt_path(exp, fold):
    return f'ck_{NOME_CURTO}_exp{exp}_fold{fold}.csv'


def fold_completo(exp, fold):
    path = ckpt_path(exp, fold)
    if not os.path.exists(path):
        return False
    df_ck = pd.read_csv(path)
    df_te = df_enem[df_enem['fold'] == fold]
    return len(df_ck.dropna(subset=['pred_total'])) >= len(df_te) * 0.95


def rodar_fold(nome, fn_inf, params, exp, fold, df_teste_fold, tipo='ZS', exemplos=None):
    temp      = params.get('temp', 0.1)
    max_chars = params.get('max_chars', None)
    n_ex      = params.get('n_exemplos', 3)
    path      = ckpt_path(exp, fold)

    if os.path.exists(path):
        df_ck     = pd.read_csv(path)
        ja_feitos = set(df_ck['index_redacao'].tolist())
        print(f'  Checkpoint fold {fold}: {len(ja_feitos)} ja processadas')
    else:
        df_ck     = pd.DataFrame(columns=['index_redacao', 'score', 'pred_total', 'feedback'])
        ja_feitos = set()

    pendentes = df_teste_fold[~df_teste_fold.index.isin(ja_feitos)]
    print(f'  Pendentes fold {fold}: {len(pendentes)}/{len(df_teste_fold)}')

    novos = []
    for idx, (i, row) in enumerate(pendentes.iterrows()):
        try:
            if tipo == 'ZS':
                prompt = gerar_prompt_zs(row['essay_limpo'], max_chars=max_chars)
                resp   = fn_inf(prompt, temp)
                fb     = None
            elif tipo == 'FS':
                prompt = gerar_prompt_fs(row['essay_limpo'], exemplos, max_chars=max_chars, n_exemplos=n_ex)
                resp   = fn_inf(prompt, temp)
                fb     = None
            else:
                prompt = gerar_prompt_cot(row['essay_limpo'], max_chars=max_chars)
                resp   = fn_inf(prompt, temp)
                fb     = extrair_feedback(resp)
            notas = extrair(resp)
            pred  = notas['Nota_Total'] if notas else None
            novos.append({'index_redacao': i, 'score': row['score'], 'pred_total': pred, 'feedback': fb})
        except:
            novos.append({'index_redacao': i, 'score': row['score'], 'pred_total': None, 'feedback': None})

        if (idx + 1) % 50 == 0:
            df_ck = pd.concat([df_ck, pd.DataFrame(novos)], ignore_index=True)
            df_ck.to_csv(path, index=False)
            novos = []
            print(f'    Checkpoint: {idx + 1 + len(ja_feitos)}/{len(df_teste_fold)}')

    if novos:
        df_ck = pd.concat([df_ck, pd.DataFrame(novos)], ignore_index=True)
        df_ck.to_csv(path, index=False)

    df_v = df_ck.dropna(subset=['pred_total'])
    print(f'  Validas fold {fold}: {len(df_v)}/{len(df_teste_fold)}')
    if len(df_v) > 0:
        return calcular_metricas(df_v['score'].tolist(), df_v['pred_total'].tolist(), f'{nome} fold{fold}')
    return None


def agregar_folds(resultados_folds, nome):
    validos = [r for r in resultados_folds if r is not None]
    if not validos:
        return None
    metricas = ['mae', 'rmse', 'qwk', 'f1']
    agregado = {'modelo': nome}
    for m in metricas:
        vals = [r[m] for r in validos if not np.isnan(r[m])]
        agregado[m]          = float(np.mean(vals)) if vals else float('nan')
        agregado[m + '_std'] = float(np.std(vals))  if vals else float('nan')
    agregado['n_folds'] = len(validos)
    print(f'\n{"="*60}')
    print(f'MEDIA 5 FOLDS — {nome}')
    print(f'{"="*60}')
    for m in metricas:
        print(f'  {m.upper():<6}: {agregado[m]:.4f} +/- {agregado[m+"_std"]:.4f}')
    print(f'  Folds validos: {agregado["n_folds"]}/{N_FOLDS}')
    print(f'{"="*60}')
    return agregado


print('Runners carregados!')

In [ ]:
# Celula 8 - Optuna (roda apenas no fold 0 para encontrar hiperparametros)

def obj_optuna_zs(trial, fn_inf):
    temp      = trial.suggest_float('temp', 0.01, 0.5)
    max_chars = trial.suggest_int('max_chars', 500, 2000, step=250)
    df_val    = df_enem[df_enem['fold'] == 1].head(10)
    yp, yt    = [], []
    for _, row in df_val.iterrows():
        try:
            prompt = gerar_prompt_zs(row['essay_limpo'], max_chars=max_chars)
            resp   = fn_inf(prompt, temp)
            notas  = extrair(resp)
            if notas and 'Nota_Total' in notas:
                yp.append(notas['Nota_Total'])
                yt.append(row['score'])
        except: pass
    if len(yp) < 5: raise optuna.TrialPruned()
    def disc(n): return np.clip(np.round(np.array(n) / 40).astype(int), 0, 25)
    try:    return cohen_kappa_score(disc(yt), disc(yp), weights='quadratic')
    except: return -1.0


def obj_optuna_fs(trial, fn_inf):
    temp       = trial.suggest_float('temp', 0.01, 0.5)
    max_chars  = trial.suggest_int('max_chars', 500, 2000, step=250)
    n_exemplos = trial.suggest_int('n_exemplos', 1, 3)
    df_val     = df_enem[df_enem['fold'] == 1].head(10)
    exemplos_v = selecionar_exemplos_fs(df_enem[df_enem['fold'] != 1])
    yp, yt     = [], []
    for _, row in df_val.iterrows():
        try:
            prompt = gerar_prompt_fs(row['essay_limpo'], exemplos_v, max_chars=max_chars, n_exemplos=n_exemplos)
            resp   = fn_inf(prompt, temp)
            notas  = extrair(resp)
            if notas and 'Nota_Total' in notas:
                yp.append(notas['Nota_Total'])
                yt.append(row['score'])
        except: pass
    if len(yp) < 5: raise optuna.TrialPruned()
    def disc(n): return np.clip(np.round(np.array(n) / 40).astype(int), 0, 25)
    try:    return cohen_kappa_score(disc(yt), disc(yp), weights='quadratic')
    except: return -1.0


def obj_optuna_cot(trial, fn_inf):
    temp      = trial.suggest_float('temp', 0.05, 0.5)
    max_chars = trial.suggest_int('max_chars', 500, 2000, step=250)
    df_val    = df_enem[df_enem['fold'] == 1].head(10)
    yp, yt    = [], []
    for _, row in df_val.iterrows():
        try:
            prompt = gerar_prompt_cot(row['essay_limpo'][:max_chars])
            resp   = fn_inf(prompt, temp)
            notas  = extrair(resp)
            if notas and 'Nota_Total' in notas:
                yp.append(notas['Nota_Total'])
                yt.append(row['score'])
        except: pass
    if len(yp) < 5: raise optuna.TrialPruned()
    def disc(n): return np.clip(np.round(np.array(n) / 40).astype(int), 0, 25)
    try:    return cohen_kappa_score(disc(yt), disc(yp), weights='quadratic')
    except: return -1.0


def rodar_optuna(obj_fn, nome, n_trials=10):
    db_name  = nome.lower().replace(' ', '_')
    storage  = f'sqlite:///optuna_{db_name}.db'
    print(f'Optuna: otimizando {nome} ({n_trials} trials)...')
    study = optuna.create_study(
        study_name=db_name,
        direction='maximize',
        pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=5),
        storage=storage,
        load_if_exists=True
    )
    trials_restantes = max(0, n_trials - len(study.trials))
    if trials_restantes > 0:
        study.optimize(obj_fn, n_trials=trials_restantes, catch=(Exception,))
    else:
        print(f'  Study ja completo ({len(study.trials)} trials). Usando resultados salvos.')
    print(f'Melhores params: {study.best_params}')
    return study.best_params


print('Optuna carregado!')

## Experimento 4 — Chain of Thought (Llama 3.3 70B)

In [ ]:
# Celula 10 - Experimento 4 (CoT)

resultados_exp4 = []

tok, m = carregar_modelo(NOME_MODELO)
fn_inf_cot = lambda prompt, temp: inf_local_cot(tok, m, prompt, temp=temp)

print('\nOtimizando hiperparametros CoT...')
params_cot = rodar_optuna(lambda t: obj_optuna_cot(t, fn_inf_cot), 'Llama 70B CoT', n_trials=8)

for fold in range(N_FOLDS):
    df_te = df_enem[df_enem['fold'] == fold].reset_index(drop=True)

    print(f'\n--- Fold {fold} | CoT ---')
    if fold_completo(4, fold):
        print(f'  Fold {fold} CoT ja completo, carregando CSV...')
        df_ck = pd.read_csv(ckpt_path(4, fold)).dropna(subset=['pred_total'])
        res = calcular_metricas(df_ck['score'].tolist(), df_ck['pred_total'].tolist(), f'Llama 70B CoT fold{fold}')
    else:
        res = rodar_fold('Llama 70B CoT', fn_inf_cot, params_cot, 4, fold, df_te, tipo='CoT')
    resultados_exp4.append(res)

liberar(m, tok)

media_exp4 = agregar_folds(resultados_exp4, 'Llama 3.3 70B (CoT)')

pd.DataFrame([r for r in resultados_exp4 if r]).to_csv(f'resultados_{NOME_CURTO}_exp4_folds.csv', index=False)
print('\nResultados Exp4 salvos!')

## Experimentos 1 e 2 — Zero-Shot e Few-Shot (Llama 3.3 70B)

O modelo e carregado uma unica vez e roda ZS e FS em sequencia antes de ser liberado.  
Para cada fold: Optuna no fold 0 define os hiperparametros, que sao reaproveitados nos demais folds.

In [ ]:
# Celula 9 - Experimentos 1 e 2 (ZS + FS no mesmo carregamento)

resultados_exp1 = []
resultados_exp2 = []

tok, m = carregar_modelo(NOME_MODELO)
fn_inf = lambda prompt, temp: inf_local(tok, m, prompt, temp=temp)

# Optuna no fold 1 (usado como validacao para encontrar hiperparametros)
print('\nOtimizando hiperparametros Zero-Shot...')
params_zs = rodar_optuna(lambda t: obj_optuna_zs(t, fn_inf), 'Llama 70B ZS', n_trials=10)

print('\nOtimizando hiperparametros Few-Shot...')
params_fs = rodar_optuna(lambda t: obj_optuna_fs(t, fn_inf), 'Llama 70B FS', n_trials=10)

# Loop pelos 5 folds
for fold in range(N_FOLDS):
    df_te = df_enem[df_enem['fold'] == fold].reset_index(drop=True)
    df_tr = df_enem[df_enem['fold'] != fold]
    exemplos = selecionar_exemplos_fs(df_tr)

    print(f'\n--- Fold {fold} | ZS ---')
    if fold_completo(1, fold):
        print(f'  Fold {fold} ZS ja completo, carregando CSV...')
        df_ck = pd.read_csv(ckpt_path(1, fold)).dropna(subset=['pred_total'])
        res = calcular_metricas(df_ck['score'].tolist(), df_ck['pred_total'].tolist(), f'Llama 70B ZS fold{fold}')
    else:
        res = rodar_fold('Llama 70B ZS', fn_inf, params_zs, 1, fold, df_te, tipo='ZS')
    resultados_exp1.append(res)

    print(f'\n--- Fold {fold} | FS ---')
    if fold_completo(2, fold):
        print(f'  Fold {fold} FS ja completo, carregando CSV...')
        df_ck = pd.read_csv(ckpt_path(2, fold)).dropna(subset=['pred_total'])
        res = calcular_metricas(df_ck['score'].tolist(), df_ck['pred_total'].tolist(), f'Llama 70B FS fold{fold}')
    else:
        res = rodar_fold('Llama 70B FS', fn_inf, params_fs, 2, fold, df_te, tipo='FS', exemplos=exemplos)
    resultados_exp2.append(res)

liberar(m, tok)

# Agregar resultados
media_exp1 = agregar_folds(resultados_exp1, 'Llama 3.3 70B (Zero-Shot)')
media_exp2 = agregar_folds(resultados_exp2, 'Llama 3.3 70B (Few-Shot)')

# Salvar resultados intermediarios
pd.DataFrame([r for r in resultados_exp1 if r]).to_csv(f'resultados_{NOME_CURTO}_exp1_folds.csv', index=False)
pd.DataFrame([r for r in resultados_exp2 if r]).to_csv(f'resultados_{NOME_CURTO}_exp2_folds.csv', index=False)
print('\nResultados Exp1 e Exp2 salvos!')

In [ ]:
# Celula 11 - Consolidacao final e graficos
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

todos = [r for r in [media_exp1, media_exp2, media_exp4] if r is not None]
df_res = pd.DataFrame(todos)

print('\n' + '='*80)
print(f'{"Modelo":<35} {"Folds":>6} {"MAE":>9} {"RMSE":>9} {"QWK":>9} {"F1":>9}')
print('-'*80)
for _, row in df_res.iterrows():
    print(
        f'{row["modelo"]:<35} '
        f'{int(row["n_folds"]):>6} '
        f'{row["mae"]:>7.3f}+/-{row["mae_std"]:.3f} '
        f'{row["rmse"]:>7.3f}+/-{row["rmse_std"]:.3f} '
        f'{row["qwk"]:>7.3f}+/-{row["qwk_std"]:.3f} '
        f'{row["f1"]:>7.3f}+/-{row["f1_std"]:.3f}'
    )
print('='*80)

df_res.to_csv(f'resultados_{NOME_CURTO}_final.csv', index=False)
print(f'\nCSV salvo: resultados_{NOME_CURTO}_final.csv')

cores = {'Zero-Shot': '#4C72B0', 'Few-Shot': '#DD8452', 'CoT': '#55A868'}
cor_lista = []
for m in df_res['modelo']:
    if 'Zero-Shot' in m: cor_lista.append(cores['Zero-Shot'])
    elif 'Few-Shot' in m: cor_lista.append(cores['Few-Shot'])
    else: cor_lista.append(cores['CoT'])

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(
    f'Llama 3.3 70B — Exp1, 2 e 4 V3\nMedia de {N_FOLDS} Folds (80/20)',
    fontsize=14, fontweight='bold'
)

for ax, (titulo, coluna) in zip(axes.flatten(), [
    ('MAE (menor = melhor)', 'mae'),
    ('RMSE (menor = melhor)', 'rmse'),
    ('QWK (maior = melhor)', 'qwk'),
    ('F1 Score (maior = melhor)', 'f1')
]):
    barras = ax.barh(df_res['modelo'], df_res[coluna], color=cor_lista, edgecolor='white')
    ax.errorbar(
        df_res[coluna], range(len(df_res)),
        xerr=df_res[coluna + '_std'],
        fmt='none', color='black', capsize=4, linewidth=1.5
    )
    for b in barras:
        w = b.get_width()
        ax.text(w + 0.002, b.get_y() + b.get_height() / 2,
                f'{w:.3f}', va='center', ha='left', fontsize=9)
    ax.set_title(titulo, fontsize=11, fontweight='bold')
    ax.set_xlabel('Valor')
    ax.grid(axis='x', alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

legenda = [Patch(color=v, label=k) for k, v in cores.items()]
fig.legend(handles=legenda, loc='lower center', ncol=3, fontsize=11,
           frameon=False, bbox_to_anchor=(0.5, -0.01))

plt.tight_layout()
plt.savefig(f'grafico_{NOME_CURTO}_final.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Grafico salvo: grafico_{NOME_CURTO}_final.png')